# 🧠 지점 단위(정류소·대여소·역) 수요 분배 모델 & 자치구 v3 — Colab 학습 파이프라인

> **이슈**: [#116](https://github.com/14EE4/weather-transit-web/issues/116) · **계획서**: `docs/MODEL_RETRAINING_PLAN.md`  
> **Phase 0 전수 감사**: 2023~2025 `217,912,809`행 스캔 완료 (`G0: PASS`)

---

## 1. 모델 개선 목적
기존 현행 모델(`v2.1-multiyear-32d`)은 서울 **25개 자치구 × 시간** 단위로 수요와 혼잡도를 계산합니다. 그 결과 **같은 자치구 안에 있는 수백 개의 버스 정류소·지하철역·따릉이 대여소가 모두 동일한 자치구 점수와 혼잡도에 묶여 계산되는 구조적 한계**가 있었습니다.

이번 재학습의 핵심 목적은 **자치구별로 계산이 묶여 부정확했던 한계를 보완하고, 개별 정류소·대여소·역별로 최대한 각각 독립적으로 수요와 혼잡도를 예측**하는 것입니다.

```text
[Stage 1: 자치구 예측 수요 (v2.1 또는 v3)]  ×  [Stage 2: 개별 지점 시간대 점유율 (M2)]  =  [개별 정류소·대여소·역 예측 이용량 & 혼잡도]
```

---

## 2. 정량적 평가 지표 및 지표 선정 이유

| 지표 | 산식 / 정의 | 역할 | 지표를 선택한 이유 |
| :--- | :--- | :---: | :--- |
| **WAPE** *(Weighted Absolute Percentage Error)* | $\frac{\sum |y_i - \hat{y}_i|}{\sum y_i} \times 100\%$ | **주 지표** | 정류소는 이용량 `0~5명`인 소형 지점과 수천 명인 환승 거점이 혼재합니다. 일반 `MAPE`는 분모가 작을 때 수백 %로 폭증해 왜곡되지만, `WAPE`는 이용량 규모로 가중 평균해 **전체 수송량 기준 실무 정확도**를 정직하게 측정합니다. |
| **MAE** *(Mean Absolute Error)* | $\frac{1}{N}\sum |y_i - \hat{y}_i|$ | **주 지표** | 지점당 오차를 원래 단위(`명`, `%p`)로 직관적 해석할 수 있고, 이상치에 과민한 RMSE와 달리 **일상 시간대 평균 오차**를 안정적으로 파악합니다. |
| **RMSE** *(Root Mean Squared Error)* | $\sqrt{\frac{1}{N}\sum(y_i - \hat{y}_i)^2}$ | **보조 지표** | 오차를 제곱해 **이용량이 큰 핵심 환승역·중앙차로 정류소의 대형 오차에 가중 페널티**를 주어 피크 시간대 혼잡도 오판을 방지합니다. |
| **Spearman 순위 상관계수 ($\rho$)** | 구×시간 내 지점별 예측 vs 실제 순위 상관 | **핵심 변별력** | 서비스 UI·추천 점수는 **"같은 구 안에서 어느 정류소가 더 붐비고 어느 역이 여유로운가"**를 가려내는 상대 서열이 핵심입니다. 자치구 일괄 점수(변별력 0) 대비 개선 효과를 직접 입증합니다. |
| **혼잡도 3등급 일치율 (%)** | 여유(`<45%`) · 보통(`45~75%`) · 혼잡(`>75%`) | **서비스 UX** | 사용자에게 최종 노출되는 3단계 혼잡 등급의 체감 일치율을 직접 검증합니다. |
| **결정계수 ($R^2$)** | 분산 설명력 | **참고 (하위 호환)** | 현행 `v2.1`($R^2 \ge 0.989$)과의 연속 비교용으로 병기하되, 규모 편차 과대평가를 피하기 위해 `WAPE·MAE`를 우선 기준으로 삼습니다. |

---

## 3. Colab 실행 전 준비사항
1. Google Drive의 `MyDrive/weather-transit-web/data/통합데이터/` 경로에 아래 데이터가 올라와 있어야 합니다:
   - `23_통합데이터/2023_통합데이터_train.csv` + `정류장_대여소_역_별_퍼센트데이터/2023_*.csv` (25개)
   - `24_통합데이터/2024_통합데이터_train.csv` + `정류장_대여소_역_별_퍼센트데이터/2024_*.csv` (25개)
   - `25_통합데이터/2025_통합데이터_test.csv` + `정류장_대여소_역_별_퍼센트데이터/2025_*.csv` (25개)
2. Colab 런타임은 **고용량 RAM 또는 기본 CPU/T4 GPU** 아무거나 사용 가능합니다 (모든 대용량 CSV는 1개 자치구씩 청크 스트리밍 처리해 OOM을 방지합니다).
3. 중간 전처리 결과(`point_priors`, 학습 샘플)는 Drive 캐시에 저장되므로, 런타임이 끊겨도 재실행 시 빠르게 이어서 학습할 수 있습니다.

In [ ]:
# [Cell 1] 환경 준비, 드라이브 마운트 및 실험 설정
import importlib, os, sys, gc, json, time, glob, pickle, warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')

# Colab 기본 미설치 패키지 자동 설치
for pkg, mod in [
    ('lightgbm', 'lightgbm'),
    ('optuna', 'optuna'),
    ('onnxmltools', 'onnxmltools'),
    ('skl2onnx', 'skl2onnx'),
    ('onnxruntime', 'onnxruntime'),
    ('scipy', 'scipy'),
    ('pyarrow', 'pyarrow'),
]:
    try:
        __import__(mod)
    except ModuleNotFoundError:
        print(f'[설치 중] {pkg} ...')
        os.system(f'pip install -q {pkg}')

importlib.invalidate_caches()
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
import onnxmltools
import onnxruntime as ort
from onnxmltools.convert.common.data_types import FloatTensorType
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Google Drive 마운트 & 경로 설정 ─────────────────────────────────────────
try:
    from google.colab import drive, files
    IN_COLAB = True
    drive.mount('/content/drive')
except ImportError:
    IN_COLAB = False
    files = None

# 사용자의 Drive 경로에 맞게 수정 가능 (기본값: MyDrive/weather-transit-web)
REPO_DIR = Path('/content/drive/MyDrive/weather-transit-web') if IN_COLAB else Path('.').resolve()
DATA_DIR = REPO_DIR / 'data' / '통합데이터'
POINT_SUBDIR = '정류장_대여소_역_별_퍼센트데이터'

WORK_CACHE_DIR = REPO_DIR / 'benchmarks' / 'cache_point_share_v3'
OUT_DIR = REPO_DIR / 'benchmarks' / f'point_share_v3_{datetime.now():%Y-%m-%d}'
RESULTS_DIR = REPO_DIR / 'benchmarks' / 'results'

for d in (WORK_CACHE_DIR, OUT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── 실험 하이퍼파라미터 / 실행 스위치 ────────────────────────────────────────
SEED = 42
TRAIN_END = '2024-09-30'
VALID_START = '2024-10-01'
VALID_END = '2024-12-31'

RUN_OPTUNA = True                 # False 면 기본 파라미터로 빠르게 검증만 수행
N_TRIALS_STAGE1 = 25              # Stage 1 자치구 v3 Optuna 탐색 횟수
N_TRIALS_STAGE2 = 20              # Stage 2 지점 점유율 Optuna 탐색 횟수
FINAL_N_ESTIMATORS = 1500
MAX_TRAIN_ROWS_PER_MODE = 1_500_000   # Stage 2 수단별 학습 행 상한 (RAM 안전장치)
MAX_VALID_ROWS_PER_MODE = 400_000     # Stage 2 수단별 검증 행 상한
FORCE_REBUILD_CACHE = False       # True 로 두면 Drive 캐시를 무시하고 CSV 부터 다시 추출

assert (DATA_DIR / '23_통합데이터' / '2023_통합데이터_train.csv').is_file(), f'경로 확인 필요: {DATA_DIR}'
assert (DATA_DIR / '24_통합데이터' / '2024_통합데이터_train.csv').is_file(), f'경로 확인 필요: {DATA_DIR}'
assert (DATA_DIR / '25_통합데이터' / '2025_통합데이터_test.csv').is_file(), f'경로 확인 필요: {DATA_DIR}'

print('✅ 환경 준비 완료')
print(f'   Python   : {sys.version.split()[0]}')
print(f'   LightGBM : {lgb.__version__} | Pandas: {pd.__version__} | NumPy: {np.__version__}')
print(f'   DATA_DIR : {DATA_DIR}')
print(f'   CACHE_DIR: {WORK_CACHE_DIR}')
print(f'   OUT_DIR  : {OUT_DIR}')

---
### [Step 2] Stage 1 데이터 로드 & 32차원 피처 엔지니어링 (공휴일 `2023~2025` + 미관측일 마스크)
- `main.py::KR_HOLIDAYS` (`2023`, `2024`, `2025` 전수)와 `DISTRICT_INFRA`, `DISTRICT_CODE_MAP`을 100% 동기화합니다.
- Phase 0 검증 보고서 JSON에서 **버스 미관측일(2023년 11일, 2024년 3일, 2025년 26일)**을 읽어 `bus_observed` 마스크를 구축합니다.

In [ ]:
# [Cell 2] 3개년 자치구 통합데이터 로드 + 32차원 피처 생성 + 미관측일 마스크
t0 = time.time()

df23 = pd.read_csv(DATA_DIR / '23_통합데이터' / '2023_통합데이터_train.csv', encoding='utf-8')
df24 = pd.read_csv(DATA_DIR / '24_통합데이터' / '2024_통합데이터_train.csv', encoding='utf-8')
df25 = pd.read_csv(DATA_DIR / '25_통합데이터' / '2025_통합데이터_test.csv', encoding='utf-8')
df = pd.concat([df23, df24, df25], ignore_index=True)
del df23, df24, df25
print(f'통합데이터 로드: {df.shape}')

# ── 1. 검증 보고서에서 수단별 미관측일 로드 (R4 마스킹용) ────────────────────
def load_unobserved_bus_dates() -> set[str]:
    missing = set()
    patterns = [
        DATA_DIR / '23_통합데이터' / POINT_SUBDIR / '검증자료' / '2023_validation_report.json',
        DATA_DIR / '24_통합데이터' / POINT_SUBDIR / '검증자료' / '검증_보고서.json',
        DATA_DIR / '25_통합데이터' / POINT_SUBDIR / '검증데이터' / '2025_데이터_검증_보고서.json',
    ]
    for p in patterns:
        if p.is_file():
            rep = json.loads(p.read_text(encoding='utf-8'))
            if '교통수단별' in rep:
                for it in rep['교통수단별']:
                    if it.get('교통수단') == '버스':
                        missing.update(it.get('미관측_날짜', []))
            else:
                missing.update(rep.get('date_coverage', {}).get('버스', {}).get('missing_days', []))
    if not missing:  # 보고서 미존재 시 Phase 0 실측 폴백
        missing = {
            '2023-01-01','2023-04-26','2023-06-05','2023-06-06','2023-06-07','2023-06-08',
            '2023-06-09','2023-06-10','2023-06-11','2023-07-11','2023-12-23',
            '2024-02-04','2024-05-28','2024-05-29',
            '2025-01-01','2025-01-02','2025-01-03','2025-01-04','2025-01-05','2025-01-06',
            '2025-01-07','2025-01-08','2025-06-30','2025-07-01','2025-07-02','2025-11-28',
            '2025-11-29','2025-11-30','2025-12-01','2025-12-02','2025-12-03','2025-12-04',
            '2025-12-20','2025-12-21','2025-12-22','2025-12-23','2025-12-26','2025-12-27',
            '2025-12-28','2025-12-31',
        }
    return missing

UNOBSERVED_BUS_DATES = load_unobserved_bus_dates()
print(f'버스 미관측일 마스크 일수: {len(UNOBSERVED_BUS_DATES)}일')

# 날짜/정렬
df['날짜_str'] = df['기준_날짜'].astype(str).str.slice(0, 10)
df['기준_날짜'] = pd.to_datetime(df['기준_날짜'])
df['기준_일시'] = pd.to_datetime(df['기준_일시'])
df = df.sort_values(['구', '기준_일시']).reset_index(drop=True)

# 기상 결측 정제
df['강수량(mm)'] = df['강수량(mm)'].fillna(0.0)
for c in ['기온(°C)', '풍속(m/s)', '습도(%)']:
    if c in df.columns:
        df[c] = df.groupby('구')[c].transform(lambda s: s.interpolate(method='linear').bfill().ffill())

# 지하철 심야 운휴 명시 0
df.loc[df['시간'].isin([1, 2, 3, 4, 5]), '지하철_승차'] = 0.0
df['따릉이_대여'] = df['따릉이_대여'].fillna(0.0)

# 보간 라벨(v2.1 호환) vs 원시 마스크 라벨(v3) 분리 보관
df['버스_승차_raw'] = df['버스_승차'].copy()
df['버스_승차_interp'] = df.groupby('구')['버스_승차'].transform(lambda s: s.interpolate(method='linear').bfill().ffill())
df['bus_observed'] = (~df['날짜_str'].isin(UNOBSERVED_BUS_DATES)) & df['버스_승차_raw'].notna()

# Stage 2 피처 조립 시에도 사용할 시간별 기상/캘린더 피처 생성 (main.py 완전 동기화)
KR_HOLIDAYS_ALL = {
    # 2023
    '2023-01-01','2023-01-21','2023-01-22','2023-01-23','2023-01-24',
    '2023-03-01','2023-05-05','2023-05-27','2023-05-29','2023-06-06',
    '2023-08-15','2023-09-28','2023-09-29','2023-09-30','2023-10-02',
    '2023-10-03','2023-10-09','2023-12-25',
    # 2024
    '2024-01-01','2024-02-09','2024-02-10','2024-02-11','2024-02-12',
    '2024-03-01','2024-04-10','2024-05-05','2024-05-06','2024-05-15',
    '2024-06-06','2024-08-15','2024-09-16','2024-09-17','2024-09-18',
    '2024-10-01','2024-10-03','2024-10-09','2024-12-25',
    # 2025
    '2025-01-01','2025-01-27','2025-01-28','2025-01-29','2025-01-30',
    '2025-03-01','2025-03-03','2025-05-05','2025-05-06','2025-06-03',
    '2025-06-06','2025-08-15','2025-10-03','2025-10-05','2025-10-06',
    '2025-10-07','2025-10-08','2025-10-09','2025-12-25',
}

df['month'] = df['기준_날짜'].dt.month
df['dayofweek'] = df['기준_날짜'].dt.dayofweek
df['is_weekend'] = (df['dayofweek'] >= 5).astype(np.float32)
df['is_holiday'] = df['날짜_str'].isin(KR_HOLIDAYS_ALL).astype(np.float32)
df['is_rush_hour'] = df['시간'].isin([8, 9, 18, 19]).astype(np.float32)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0).astype(np.float32)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0).astype(np.float32)
df['hour_sin'] = np.sin(2 * np.pi * df['시간'] / 24.0).astype(np.float32)
df['hour_cos'] = np.cos(2 * np.pi * df['시간'] / 24.0).astype(np.float32)

df['temperature'] = df['기온(°C)'].astype(np.float32)
df['wind_speed'] = df['풍속(m/s)'].clip(lower=0.1).astype(np.float32)
df['humidity'] = df['습도(%)'].astype(np.float32)
df['precipitation'] = df['강수량(mm)'].astype(np.float32)
df['is_raining'] = (df['precipitation'] > 0.0).astype(np.float32)
df['rain_intensity'] = np.select(
    [df['precipitation'] <= 0.0, df['precipitation'] < 3.0, df['precipitation'] <= 10.0],
    [0, 1, 2], default=3,
).astype(np.float32)
_v = df['wind_speed'] ** 0.16
df['sensible_temp'] = (13.12 + 0.6215 * df['temperature'] - 11.37 * _v + 0.3965 * df['temperature'] * _v).astype(np.float32)
df['discomfort_index'] = (1.8 * df['temperature'] - 0.55 * (1 - df['humidity'] / 100.0) * (1.8 * df['temperature'] - 26.0) + 32.0).astype(np.float32)

DISTRICT_CODE_MAP = {
    '강남구': 0, '강동구': 1, '강북구': 2, '강서구': 3, '관악구': 4,
    '광진구': 5, '구로구': 6, '금천구': 7, '노원구': 8, '도봉구': 9,
    '동대문구': 10, '동작구': 11, '마포구': 12, '서대문구': 13, '서초구': 14,
    '성동구': 15, '성북구': 16, '송파구': 17, '양천구': 18, '영등포구': 19,
    '용산구': 20, '은평구': 21, '종로구': 22, '중구': 23, '중랑구': 24,
}
DISTRICT_INFRA = {
    '강남구': (34.0, 552.0, 172.0, 1.48), '강동구': (16.0, 398.0, 116.0, 0.86),
    '강북구': (13.0, 428.0, 90.0, 0.72),  '강서구': (23.0, 628.0, 212.0, 0.92),
    '관악구': (9.0, 492.0, 132.0, 0.88),  '광진구': (11.0, 298.0, 120.0, 0.96),
    '구로구': (15.0, 524.0, 138.0, 1.08), '금천구': (5.0, 368.0, 96.0, 1.02),
    '노원구': (18.0, 558.0, 158.0, 0.82), '도봉구': (8.0, 402.0, 102.0, 0.70),
    '동대문구': (16.0, 498.0, 130.0, 1.05), '동작구': (17.0, 448.0, 118.0, 0.92),
    '마포구': (18.0, 574.0, 168.0, 1.28), '서대문구': (10.0, 462.0, 110.0, 0.95),
    '서초구': (27.0, 615.0, 156.0, 1.36), '성동구': (17.0, 442.0, 128.0, 1.12),
    '성북구': (14.0, 602.0, 124.0, 0.85), '송파구': (30.0, 645.0, 228.0, 1.22),
    '양천구': (10.0, 388.0, 136.0, 0.86), '영등포구': (21.0, 516.0, 184.0, 1.42),
    '용산구': (15.0, 356.0, 106.0, 1.18), '은평구': (16.0, 518.0, 142.0, 0.76),
    '종로구': (19.0, 386.0, 126.0, 1.35), '중구': (26.0, 248.0, 114.0, 1.46),
    '중랑구': (12.0, 378.0, 122.0, 0.76),
}
df['district_code'] = df['구'].map(DISTRICT_CODE_MAP).astype(np.float32)
infra_df = pd.DataFrame(DISTRICT_INFRA, index=['district_subway_stations', 'district_bus_stops', 'district_bike_racks', 'commercial_density']).T
df = df.merge(infra_df, left_on='구', right_index=True, how='left')

# 시계열 래그 (보간 버스 라벨 기준으로 계산 후 마스킹 비교)
df['버스_승차'] = df['버스_승차_interp']
LAG_TARGETS = {'bike': '따릉이_대여', 'bus': '버스_승차', 'sub': '지하철_승차'}
for mode, col in LAG_TARGETS.items():
    df[f'{col}_lag_1h'] = df.groupby('구')[col].shift(1)
    df[f'{col}_roll_3h'] = df.groupby('구')[col].transform(lambda x: x.rolling(3, min_periods=1).mean())
    df[f'{col}_lag_1h'] = df[f'{col}_lag_1h'].fillna(df[f'{col}_roll_3h']).astype(np.float32)
    df[f'{col}_roll_3h'] = df[f'{col}_roll_3h'].astype(np.float32)

df['temp_diff_1h'] = df.groupby('구')['temperature'].diff().fillna(0.0).astype(np.float32)
df['rain_diff_1h'] = df.groupby('구')['precipitation'].diff().fillna(0.0).astype(np.float32)

prev = df[['구', '기준_일시'] + list(LAG_TARGETS.values())].copy()
prev['기준_일시'] = prev['기준_일시'] + pd.Timedelta(days=7)
prev = prev.rename(columns={
    '따릉이_대여': 'same_day_last_week_bike',
    '버스_승차': 'same_day_last_week_bus',
    '지하철_승차': 'same_day_last_week_sub',
})
df = df.merge(prev, on=['구', '기준_일시'], how='left')
for mode, col in LAG_TARGETS.items():
    dst = f'same_day_last_week_{mode}'
    df[dst] = df[dst].fillna(df[f'{col}_lag_1h']).fillna(0.0).astype(np.float32)

FEATURE_COLS_32 = [
    'month_sin', 'month_cos', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend', 'is_holiday', 'is_rush_hour',
    'temperature', 'wind_speed', 'humidity', 'precipitation', 'is_raining', 'rain_intensity', 'sensible_temp', 'discomfort_index',
    'district_code', 'district_subway_stations', 'district_bus_stops', 'district_bike_racks', 'commercial_density',
    '따릉이_대여_lag_1h', '버스_승차_lag_1h', '지하철_승차_lag_1h',
    '따릉이_대여_roll_3h', '버스_승차_roll_3h', '지하철_승차_roll_3h',
    'same_day_last_week_bike', 'same_day_last_week_bus', 'same_day_last_week_sub',
    'temp_diff_1h', 'rain_diff_1h',
]
assert len(FEATURE_COLS_32) == 32 and int(df[FEATURE_COLS_32].isna().sum().sum()) == 0

train_mask = df['날짜_str'] <= TRAIN_END
val_mask = (df['날짜_str'] >= VALID_START) & (df['날짜_str'] <= VALID_END)
test_mask = df['날짜_str'] > VALID_END

print(f'train: {int(train_mask.sum()):,} | valid: {int(val_mask.sum()):,} | test: {int(test_mask.sum()):,} | 완료 ({time.time()-t0:.1f}s)')

---
### [Step 3] Stage 1 — 자치구 수요 모델 `v2.1(보간)` vs `v3(미관측일 마스킹)` 비교 및 게이트 G3
- 공정 평가를 위해 **2025 Test 평가는 반드시 관측 가능한 날짜 마스크(`bus_observed`) 위에서 동일 기준**으로 계산합니다.
- 게이트 G3(3수단 중 2수단 이상 MAE 개선, 나머지 악화 `2%` 이내)를 통과하면 Stage 1에 `v3`를 채택하고, 미달 시 안정적인 기존 파라미터(`v2.1` 방식)를 유지합니다.

In [ ]:
# [Cell 3] Stage 1 자치구 모델 비교 학습 + 2025 자치구 예측값 테이블 확보
def calc_regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_t = np.asarray(y_true, dtype=np.float64)
    y_p = np.clip(np.asarray(y_pred, dtype=np.float64), 0.0, None)
    denom = float(np.sum(np.abs(y_t)))
    wape = float(np.sum(np.abs(y_t - y_p)) / denom * 100.0) if denom > 0 else 0.0
    return {
        'wape': wape,
        'mae': float(mean_absolute_error(y_t, y_p)),
        'rmse': float(np.sqrt(mean_squared_error(y_t, y_p))),
        'r2': float(r2_score(y_t, y_p)),
    }

# 현행 v2.1 동결 파라미터 (feature_schema.json)
V21_PARAMS = {
    'bike': {'learning_rate': 0.09737213718467148, 'num_leaves': 115, 'max_depth': 10, 'subsample': 0.6042083739397852, 'colsample_bytree': 0.8354334394368443, 'reg_alpha': 4.417464355796284, 'reg_lambda': 0.38728221291236736},
    'bus': {'learning_rate': 0.08197452993703759, 'num_leaves': 96, 'max_depth': 10, 'subsample': 0.8701829831827598, 'colsample_bytree': 0.7447529351518585, 'reg_alpha': 6.673494947922648, 'reg_lambda': 3.025329257064882},
    'subway': {'learning_rate': 0.061872344269327376, 'num_leaves': 115, 'max_depth': 10, 'subsample': 0.7495170316026032, 'colsample_bytree': 0.6186536863498528, 'reg_alpha': 6.251880404867598, 'reg_lambda': 1.1291269775287658},
}

TARGETS_S1 = {'bike': ('따릉이_대여', '따릉이_대여'), 'bus': ('버스_승차_interp', '버스_승차_raw'), 'subway': ('지하철_승차', '지하철_승차')}
X_s1_tr = df.loc[train_mask, FEATURE_COLS_32].astype(np.float32)
X_s1_va = df.loc[val_mask, FEATURE_COLS_32].astype(np.float32)
X_s1_te = df.loc[test_mask, FEATURE_COLS_32].astype(np.float32)

stage1_models = {}
stage1_report = {}
stage1_test_df = df.loc[test_mask, ['날짜_str', '시간', '구']].copy().reset_index(drop=True)

for mode, (col_interp, col_raw) in TARGETS_S1.items():
    print(f'\n=== [Stage 1 · {mode.upper()}] v2.1(보간) vs v3(미관측 마스킹) ===')
    eval_obs = np.ones(int(test_mask.sum()), dtype=bool)
    if mode == 'bus':
        eval_obs = df.loc[test_mask, 'bus_observed'].to_numpy()
    y_te_eval = df.loc[test_mask, col_raw].to_numpy(dtype=np.float64)[eval_obs]

    # 1) Baseline(v2.1 방식: 보간 데이터 + V21_PARAMS)
    p_base = dict(V21_PARAMS[mode], objective='regression', metric='rmse', n_estimators=FINAL_N_ESTIMATORS, random_state=SEED, n_jobs=-1, verbose=-1)
    m_v21 = lgb.LGBMRegressor(**p_base)
    m_v21.fit(X_s1_tr, df.loc[train_mask, col_interp].to_numpy(dtype=np.float64),
              eval_set=[(X_s1_va, df.loc[val_mask, col_interp].to_numpy(dtype=np.float64))],
              callbacks=[lgb.early_stopping(50, verbose=False)])
    pred_v21 = np.clip(m_v21.predict(X_s1_te), 0.0, None)
    met_v21 = calc_regression_metrics(y_te_eval, pred_v21[eval_obs])

    # 2) Candidate(v3 방식: 미관측 마스킹 + 선택적 Optuna)
    tr_obs = train_mask & (df['bus_observed'] if mode == 'bus' else True)
    va_obs = val_mask & (df['bus_observed'] if mode == 'bus' else True)
    X_tr_v3 = df.loc[tr_obs, FEATURE_COLS_32].astype(np.float32)
    y_tr_v3 = df.loc[tr_obs, col_raw].to_numpy(dtype=np.float64)
    X_va_v3 = df.loc[va_obs, FEATURE_COLS_32].astype(np.float32)
    y_va_v3 = df.loc[va_obs, col_raw].to_numpy(dtype=np.float64)

    best_p = dict(V21_PARAMS[mode])
    if RUN_OPTUNA and mode == 'bus':  # 마스킹 변경이 있는 버스는 재탐색, 타 수단은 옵션 적용
        def obj_s1(trial):
            cand = {
                'objective': 'regression', 'metric': 'rmse', 'n_estimators': 600,
                'learning_rate': trial.suggest_float('learning_rate', 0.015, 0.1, log=True),
                'num_leaves': trial.suggest_int('num_leaves', 48, 127),
                'max_depth': trial.suggest_int('max_depth', 7, 10),
                'subsample': trial.suggest_float('subsample', 0.6, 0.95),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-2, 10.0, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-2, 10.0, log=True),
                'random_state': SEED, 'n_jobs': -1, 'verbose': -1,
            }
            mm = lgb.LGBMRegressor(**cand)
            mm.fit(X_tr_v3, y_tr_v3, eval_set=[(X_va_v3, y_va_v3)], callbacks=[lgb.early_stopping(30, verbose=False)])
            return float(np.sqrt(mean_squared_error(y_va_v3, np.clip(mm.predict(X_va_v3), 0, None))))
        study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(obj_s1, n_trials=N_TRIALS_STAGE1)
        best_p = dict(study.best_params)

    m_v3 = lgb.LGBMRegressor(**best_p, objective='regression', metric='rmse', n_estimators=FINAL_N_ESTIMATORS, random_state=SEED, n_jobs=-1, verbose=-1)
    m_v3.fit(X_tr_v3, y_tr_v3, eval_set=[(X_va_v3, y_va_v3)], callbacks=[lgb.early_stopping(50, verbose=False)])
    pred_v3 = np.clip(m_v3.predict(X_s1_te), 0.0, None)
    met_v3 = calc_regression_metrics(y_te_eval, pred_v3[eval_obs])

    win = 'v3' if met_v3['mae'] < met_v21['mae'] else 'v2.1'
    stage1_models[mode] = m_v3 if win == 'v3' else m_v21
    stage1_test_df[f'pred_{mode}'] = pred_v3 if win == 'v3' else pred_v21
    stage1_report[mode] = {'v2.1': met_v21, 'v3': met_v3, 'winner': win}
    print(f"  v2.1 | WAPE={met_v21['wape']:.2f}% MAE={met_v21['mae']:.2f} RMSE={met_v21['rmse']:.2f} R²={met_v21['r2']:.4f}")
    print(f"  v3   | WAPE={met_v3['wape']:.2f}% MAE={met_v3['mae']:.2f} RMSE={met_v3['rmse']:.2f} R²={met_v3['r2']:.4f}  → 승자: {win}")

# 게이트 G3 판정
mae_improved = sum(1 for m in stage1_report.values() if m['v3']['mae'] < m['v2.1']['mae'])
max_degrade = max((m['v3']['mae'] - m['v2.1']['mae']) / m['v2.1']['mae'] for m in stage1_report.values())
g3_pass = (mae_improved >= 2) and (max_degrade <= 0.02)
print(f'\n[게이트 G3 판정] {"PASS (v3 채택)" if g3_pass else "KEEP_V21 (현행 v2.1 유지, 개별로 더 좋은 모델 사용)"} | MAE 개선 {mae_improved}/3, 최대악화 {max_degrade*100:+.2f}%')

# Stage 2 에서 결합해 쓸 시간·구 단위 기상/캘린더 보조 테이블 준비
WEATHER_CAL_COLS = ['날짜_str', '시간', '구', 'month_sin', 'month_cos', 'hour_sin', 'hour_cos',
                    'dayofweek', 'is_weekend', 'is_holiday', 'is_rush_hour',
                    'temperature', 'precipitation', 'wind_speed', 'humidity']
wx_cal_df = df[WEATHER_CAL_COLS].drop_duplicates(['날짜_str', '시간', '구']).reset_index(drop=True)
print(f'기상·캘린더 조인 테이블: {wx_cal_df.shape}')
gc.collect()

---
### [Step 4] Stage 2 전처리 — 학습구간(`<= 2024-09-30`) 전용 지점 사전 프로필(Point Priors) 추출
- **미래 정보 누수(R1) 원천 차단**: 원천 CSV의 `연간_점유율(%)`, `지점_연간총이용량` 등은 **절대 사용하지 않고**, 오직 학습구간(`<= 2024-09-30`) 데이터만 스트리밍해 각 정류소·대여소·역의 고유 프로필을 산출합니다.
- **상위 K 확정 (Phase 0 실측 반영)**:
  - 🚇 **지하철**: 전체 `258`역 전수 유지 (`__OTHER__` 불필요할 정도로 작음)
  - 🚲 **따릉이 / 🚌 버스**: 구·수단별 학습구간 누적 이용량 **상위 95% (`K95`)** 지점 유지 + 나머지는 `__OTHER__` 버킷으로 합산 (분배 합 100% 보존)
- **Drive 캐시**: 추출 결과는 `stage2_priors_cache.pkl` 로 저장되어 재실행 시 스캔 없이 즉시 로드됩니다.

In [ ]:
# [Cell 4] 학습구간(<=2024-09-30) 전용 지점 사전 프로필 & K95 지점 집합 추출
PRIORS_CACHE_PATH = WORK_CACHE_DIR / 'stage2_priors_cache.pkl'
MODES_KO = {'따릉이': 'bike', '버스': 'bus', '지하철': 'subway'}
KO_FROM_MODE = {v: k for k, v in MODES_KO.items()}

# 기상/주말/출퇴근lookup용 최소 테이블 (학습구간 날짜만)
_cal_lookup = wx_cal_df[wx_cal_df['날짜_str'] <= TRAIN_END][['날짜_str', '시간', '구', 'is_weekend', 'is_rush_hour', 'precipitation']].copy()
_cal_lookup['is_rain'] = (_cal_lookup['precipitation'] > 0.0).astype(np.int8)
_cal_lookup['is_weekend'] = _cal_lookup['is_weekend'].astype(np.int8)
_cal_lookup['is_rush_hour'] = _cal_lookup['is_rush_hour'].astype(np.int8)

# 학습구간 중 수단별 관측 가능 일수 (자치구별 동일)
_train_dates = wx_cal_df[wx_cal_df['날짜_str'] <= TRAIN_END][['날짜_str']].drop_duplicates()
TRAIN_OBS_DAYS = {
    '지하철': len(_train_dates),
    '따릉이': len(_train_dates),
    '버스': int((~_train_dates['날짜_str'].isin(UNOBSERVED_BUS_DATES)).sum()),
}
# 관측 가능한 총 시간 셀 수 (정류소 평균 점유율 분모용: 무이용 0% 포함!)
TRAIN_OBS_HOURS = {
    '지하철': TRAIN_OBS_DAYS['지하철'] * 18,  # 06~23시
    '따릉이': TRAIN_OBS_DAYS['따릉이'] * 23,  # 01~23시
    '버스':   TRAIN_OBS_DAYS['버스'] * 23,    # 01~23시
}
_train_slot_cnt = {}
for _mk in ['지하철', '따릉이', '버스']:
    _sub = _cal_lookup.copy()
    if _mk == '지하철':
        _sub = _sub[_sub['시간'] >= 6]
    if _mk == '버스':
        _sub = _sub[~_sub['날짜_str'].isin(UNOBSERVED_BUS_DATES)]
    # 자치구는 25개 동일 날짜/시간 격자이므로 강남구 1개 기준으로 시간/조건별 슬롯 수 산정
    _one = _sub[_sub['구'] == '강남구']
    _train_slot_cnt[_mk] = {
        'total': len(_one),
        'wh': _one.groupby(['is_weekend', '시간']).size().to_dict(),
        'rush1': int((_one['is_rush_hour'] == 1).sum()),
        'rush0': int((_one['is_rush_hour'] == 0).sum()),
    }
_rain_cnt_by_dist = (
    _cal_lookup.groupby(['구', 'is_rain']).size().unstack(fill_value=0).to_dict(orient='index')
)

if PRIORS_CACHE_PATH.is_file() and not FORCE_REBUILD_CACHE:
    print(f'📦 Drive 캐시에서 Point Priors 로드: {PRIORS_CACHE_PATH.name}')
    with open(PRIORS_CACHE_PATH, 'rb') as f:
        priors_bundle = pickle.load(f)
else:
    print('🔍 2023~2024-09-30 원천 지점 CSV 청크 스캔 시작 (미래 데이터 0% 제외)...')
    t_s = time.time()
    # 1단계: 학습구간 총 이용량 합계 → K95 선정
    train_files_23 = sorted(glob.glob(str(DATA_DIR / '23_통합데이터' / POINT_SUBDIR / '2023_*.csv')))
    train_files_24 = sorted(glob.glob(str(DATA_DIR / '24_통합데이터' / POINT_SUBDIR / '2024_*.csv')))
    assert len(train_files_23) == 25 and len(train_files_24) == 25

    pt_usage_parts = []
    # 지점×(주말,시간), 출퇴근, 강수, p95 근사를 동시에 구별 파일 단위로 적재
    wh_sum_parts, rush_sum_parts, rain_sum_parts, p95_parts = [], [], [], []
    dist_name_map = {}

    for idx_f, fpath in enumerate(train_files_23 + train_files_24, 1):
        # 구 이름 추출 (예: 2023_강남구.csv -> 강남구)
        dist = Path(fpath).stem.split('_', 1)[1]
        wx_d = _cal_lookup[_cal_lookup['구'] == dist][['날짜_str', '시간', 'is_weekend', 'is_rush_hour', 'is_rain']]
        chunks_kept = []
        for ch in pd.read_csv(
            fpath,
            usecols=['기준_날짜', '시간', '행정구', '교통수단', '지점ID', '지점명', '이용량', '시간대_점유율(%)'],
            dtype={'지점ID': str}, encoding='utf-8', chunksize=1_000_000,
        ):
            ch = ch[ch['기준_날짜'] <= TRAIN_END]
            if ch.empty:
                continue
            # 버스 미관측일 제외
            ch = ch[~((ch['교통수단'] == '버스') & (ch['기준_날짜'].isin(UNOBSERVED_BUS_DATES)))]
            if ch.empty:
                continue
            chunks_kept.append(ch)

        if not chunks_kept:
            continue
        df_d = pd.concat(chunks_kept, ignore_index=True)
        # 최신 지점명 기록
        dist_name_map.update(df_d.drop_duplicates('지점ID').set_index('지점ID')['지점명'].to_dict())

        pt_usage_parts.append(
            df_d.groupby(['행정구', '교통수단', '지점ID'], sort=False)
            .agg(usage_sum=('이용량', 'sum'), share_sum=('시간대_점유율(%)', 'sum'))
            .reset_index()
        )
        # 기상/주말 속성 조인 후 조건부 점유율 합계 누적
        df_d = df_d.merge(wx_d, left_on=['기준_날짜', '시간'], right_on=['날짜_str', '시간'], how='left')
        wh_sum_parts.append(
            df_d.groupby(['행정구', '교통수단', '지점ID', 'is_weekend', '시간'], sort=False)['시간대_점유율(%)'].sum().reset_index()
        )
        rush_sum_parts.append(
            df_d.groupby(['행정구', '교통수단', '지점ID', 'is_rush_hour'], sort=False)['시간대_점유율(%)'].sum().reset_index()
        )
        rain_sum_parts.append(
            df_d.groupby(['행정구', '교통수단', '지점ID', 'is_rain'], sort=False)['시간대_점유율(%)'].sum().reset_index()
        )
        # 학습구간 내 관측된 양수 이용량들의 p95/max (혼잡도 100% 상한 계산용)
        p95_parts.append(
            df_d.groupby(['행정구', '교통수단', '지점ID'], sort=False)['이용량'].quantile(0.95).reset_index()
        )
        if idx_f % 10 == 0 or idx_f == 50:
            print(f'  [{idx_f:2d}/50] {Path(fpath).name} 완료 ({time.time()-t_s:.0f}s)')

    # 전체 합산
    pt_tot = pd.concat(pt_usage_parts, ignore_index=True).groupby(['행정구', '교통수단', '지점ID'], as_index=False).sum(numeric_only=True)
    wh_tot = pd.concat(wh_sum_parts, ignore_index=True).groupby(['행정구', '교통수단', '지점ID', 'is_weekend', '시간'], as_index=False)['시간대_점유율(%)'].sum()
    rush_tot = pd.concat(rush_sum_parts, ignore_index=True).groupby(['행정구', '교통수단', '지점ID', 'is_rush_hour'], as_index=False)['시간대_점유율(%)'].sum()
    rain_tot = pd.concat(rain_sum_parts, ignore_index=True).groupby(['행정구', '교통수단', '지점ID', 'is_rain'], as_index=False)['시간대_점유율(%)'].sum()
    p95_tot = pd.concat(p95_parts, ignore_index=True).groupby(['행정구', '교통수단', '지점ID'], as_index=False)['이용량'].median().rename(columns={'이용량': 'point_p95'})

    # 구·수단별 K95 유지 지점 확정 (지하철=전수, 따릉이/버스=K95)
    kept_points = {}  # (행정구, 교통수단) -> set(지점ID)
    for (dist, mk), g in pt_tot.groupby(['행정구', '교통수단']):
        g_s = g.sort_values('usage_sum', ascending=False).reset_index(drop=True)
        if mk == '지하철':
            kept_points[(dist, mk)] = set(g_s['지점ID'])
        else:
            cum = g_s['usage_sum'].cumsum() / max(float(g_s['usage_sum'].sum()), 1.0)
            k = int(np.searchsorted(cum.to_numpy(), 0.95) + 1)
            kept_points[(dist, mk)] = set(g_s.iloc[:k]['지점ID'])

    # __OTHER__ 재집계 및 기본 점유율/탄력성/첨두비율 산출
    def map_bucket(df_in):
        out = df_in.copy()
        keep_mask = [pid in kept_points.get((d, m), set()) for d, m, pid in zip(out['행정구'], out['교통수단'], out['지점ID'])]
        out.loc[~np.array(keep_mask), '지점ID'] = '__OTHER__'
        return out

    pt_b = map_bucket(pt_tot).groupby(['행정구', '교통수단', '지점ID'], as_index=False).sum(numeric_only=True)
    wh_b = map_bucket(wh_tot).groupby(['행정구', '교통수단', '지점ID', 'is_weekend', '시간'], as_index=False).sum(numeric_only=True)
    rush_b = map_bucket(rush_tot).groupby(['행정구', '교통수단', '지점ID', 'is_rush_hour'], as_index=False).sum(numeric_only=True)
    rain_b = map_bucket(rain_tot).groupby(['행정구', '교통수단', '지점ID', 'is_rain'], as_index=False).sum(numeric_only=True)
    p95_b = map_bucket(p95_tot).groupby(['행정구', '교통수단', '지점ID'], as_index=False)['point_p95'].sum()

    # 관측 가능 시간 수로 나눠 '무이용(0%) 포함 진짜 평균 점유율' 산출
    pt_b['total_slots'] = pt_b['교통수단'].map(lambda m: _train_slot_cnt[m]['total'])
    pt_b['point_prior_base_share'] = (pt_b['share_sum'] / pt_b['total_slots']).astype(np.float32)

    # 출퇴근 집중도 (rush_ratio)
    r_piv = rush_b.pivot_table(index=['행정구', '교통수단', '지점ID'], columns='is_rush_hour', values='시간대_점유율(%)', fill_value=0.0).reset_index()
    if 1 not in r_piv.columns: r_piv[1] = 0.0
    if 0 not in r_piv.columns: r_piv[0] = 0.0
    r_piv['r1_avg'] = r_piv[1] / r_piv['교통수단'].map(lambda m: max(_train_slot_cnt[m]['rush1'], 1))
    r_piv['r0_avg'] = r_piv[0] / r_piv['교통수단'].map(lambda m: max(_train_slot_cnt[m]['rush0'], 1))
    r_piv['point_prior_rush_ratio'] = np.clip((r_piv['r1_avg'] + 0.05) / (r_piv['r0_avg'] + 0.05), 0.2, 5.0).astype(np.float32)

    # 우천 탄력성 (rain_elasticity)
    rn_piv = rain_b.pivot_table(index=['행정구', '교통수단', '지점ID'], columns='is_rain', values='시간대_점유율(%)', fill_value=0.0).reset_index()
    if 1 not in rn_piv.columns: rn_piv[1] = 0.0
    if 0 not in rn_piv.columns: rn_piv[0] = 0.0
    rn_piv['n_rain'] = rn_piv['행정구'].map(lambda d: max(_rain_cnt_by_dist.get(d, {}).get(1, 1), 1))
    rn_piv['n_dry'] = rn_piv['행정구'].map(lambda d: max(_rain_cnt_by_dist.get(d, {}).get(0, 1), 1))
    rn_piv['point_prior_rain_elasticity'] = np.clip(
        ((rn_piv[1] / rn_piv['n_rain']) + 0.05) / ((rn_piv[0] / rn_piv['n_dry']) + 0.05), 0.2, 5.0
    ).astype(np.float32)

    static_priors = (
        pt_b[['행정구', '교통수단', '지점ID', 'point_prior_base_share']]
        .merge(r_piv[['행정구', '교통수단', '지점ID', 'point_prior_rush_ratio']], on=['행정구', '교통수단', '지점ID'], how='left')
        .merge(rn_piv[['행정구', '교통수단', '지점ID', 'point_prior_rain_elasticity']], on=['행정구', '교통수단', '지점ID'], how='left')
        .merge(p95_b[['행정구', '교통수단', '지점ID', 'point_p95']], on=['행정구', '교통수단', '지점ID'], how='left')
    )
    static_priors['point_p95'] = static_priors['point_p95'].fillna(10.0).clip(lower=5.0).astype(np.float32)

    # (주말, 시간대) 조건부 평균 점유율 + 베이지안 스무딩 (m=10)
    wh_b['slot_cnt'] = [max(_train_slot_cnt[m]['wh'].get((w, h), 1), 1) for m, w, h in zip(wh_b['교통수단'], wh_b['is_weekend'], wh_b['시간'])]
    wh_b = wh_b.merge(static_priors[['행정구', '교통수단', '지점ID', 'point_prior_base_share']], on=['행정구', '교통수단', '지점ID'], how='left')
    m_smooth = 10.0
    wh_b['point_prior_dow_hour_share'] = (
        (wh_b['시간대_점유율(%)'] + m_smooth * wh_b['point_prior_base_share']) / (wh_b['slot_cnt'] + m_smooth)
    ).astype(np.float32)

    # 콜드스타트 폴백(구·수단 평균)
    cold_fallback = static_priors.groupby(['행정구', '교통수단'], as_index=False)['point_prior_base_share'].mean()

    priors_bundle = {
        'kept_points': kept_points,
        'static_priors': static_priors,
        'dow_hour_priors': wh_b[['행정구', '교통수단', '지점ID', 'is_weekend', '시간', 'point_prior_dow_hour_share']],
        'cold_fallback': cold_fallback,
        'dist_name_map': dist_name_map,
    }
    with open(PRIORS_CACHE_PATH, 'wb') as f:
        pickle.dump(priors_bundle, f)
    print(f'✅ Point Priors 캐시 저장 완료: {PRIORS_CACHE_PATH.name} ({time.time()-t_s:.0f}s)')

for mk, en in MODES_KO.items():
    n_k = sum(len(v) for (d, m), v in priors_bundle['kept_points'].items() if m == mk)
    print(f'  - {mk:3s}({en:6s}) 유지 지점 수: {n_k:,}개 (+ __OTHER__ 버킷)')

---
### [Step 5] Stage 2 학습/검증 데이터셋 구축 (16차원 순수 수치형 피처 & 누수 가드)
- **금지 컬럼 가드**: 미래 연간 정보(`연간_점유율(%)`, `지점_연간총이용량`, `행정구_연간총이용량`)와 타깃 자체(`시간대_점유율(%)`, `이용량`, `행정구_시간대총이용량`)는 피처에 **절대 포함하지 않음을 assert로 강제**합니다.
- 각 자치구별로 관측 가능 시간 격자에 유지 지점들을 **밀집화(무이용 0% 포함)** 한 뒤, 수단별 층화 샘플링으로 `train`/`valid` Parquet을 구축합니다.

In [ ]:
# [Cell 5] Stage 2 16차원 학습/검증 데이터셋 구축 (구별 밀집화 + 층화 샘플링 + Drive 캐시)
POINT_SHARE_FEATURE_COLS = [
    # 1. 캘린더·시간 (8)
    'month_sin', 'month_cos', 'hour_sin', 'hour_cos',
    'dayofweek', 'is_weekend', 'is_holiday', 'is_rush_hour',
    # 2. 기상 (4)
    'temperature', 'precipitation', 'wind_speed', 'humidity',
    # 3. 학습구간 전용 지점 사전 프로필 (4) — 누수 0%
    'point_prior_base_share', 'point_prior_dow_hour_share',
    'point_prior_rush_ratio', 'point_prior_rain_elasticity',
]
FORBIDDEN_LEAKAGE_COLS = {
    '연간_점유율(%)', '지점_연간총이용량', '행정구_연간총이용량',
    '행정구_시간대총이용량', '시간대_점유율(%)', '이용량',
}
assert len(POINT_SHARE_FEATURE_COLS) == 16
assert not (set(POINT_SHARE_FEATURE_COLS) & FORBIDDEN_LEAKAGE_COLS), '🚨 데이터 누수 컬럼이 피처에 포함되었습니다!'
print(f'✅ 누수 가드 통과 · Stage 2 입력 차원 = {len(POINT_SHARE_FEATURE_COLS)}차원')


def build_dense_district_mode_df(
    raw_df: pd.DataFrame, dist: str, mk: str, kept_set: set[str],
    date_min: str, date_max: str, is_eval: bool = False
) -> pd.DataFrame:
    """단일 자치구·단일 수단의 희소 데이터를 관측일 기준 밀집화(무이용 0% 포함)하고 16차원 피처를 결합한다."""
    sub = raw_df[(raw_df['교통수단'] == mk) & (raw_df['기준_날짜'] >= date_min) & (raw_df['기준_날짜'] <= date_max)].copy()
    if mk == '버스':
        sub = sub[~sub['기준_날짜'].isin(UNOBSERVED_BUS_DATES)]
    if sub.empty:
        return pd.DataFrame()

    # 유지 지점 외의 지점은 '__OTHER__' 로 합산 (is_eval=True 일 때 콜드스타트 식별 유지 여부에 따라 분기 가능)
    sub['bucket_id'] = np.where(sub['지점ID'].isin(kept_set), sub['지점ID'], '__OTHER__')
    agg = sub.groupby(['기준_날짜', '시간', 'bucket_id'], sort=False, as_index=False).agg(
        target_share=('시간대_점유율(%)', 'sum'),
        actual_usage=('이용량', 'sum'),
    )

    # 관측 가능한 (기준_날짜, 시간) 그리드 × 모든 버킷 카테시안 곱 (무이용 0% 생성)
    wx_sub = wx_cal_df[(wx_cal_df['구'] == dist) & (wx_cal_df['날짜_str'] >= date_min) & (wx_cal_df['날짜_str'] <= date_max)].copy()
    if mk == '지하철':
        wx_sub = wx_sub[wx_sub['시간'] >= 6]
    if mk == '버스':
        wx_sub = wx_sub[~wx_sub['날짜_str'].isin(UNOBSERVED_BUS_DATES)]
    wx_sub = wx_sub.rename(columns={'날짜_str': '기준_날짜'})

    buckets = sorted(kept_set | ({'__OTHER__'} if mk != '지하철' and ('__OTHER__' in agg['bucket_id'].values) else set()))
    grid = wx_sub[['기준_날짜', '시간']].assign(_k=1).merge(
        pd.DataFrame({'bucket_id': buckets, '_k': 1}), on='_k', how='inner'
    ).drop(columns='_k')

    dense = grid.merge(agg, on=['기준_날짜', '시간', 'bucket_id'], how='left')
    dense['target_share'] = dense['target_share'].fillna(0.0).astype(np.float32)
    dense['actual_usage'] = dense['actual_usage'].fillna(0.0).astype(np.float32)

    # 16차원 피처 조인: 기상/캘린더(12) + 지점 프로필(4)
    dense = dense.merge(wx_sub, on=['기준_날짜', '시간'], how='left')

    sp = priors_bundle['static_priors']
    sp_dm = sp[(sp['행정구'] == dist) & (sp['교통수단'] == mk)].drop(columns=['행정구', '교통수단'])
    dp = priors_bundle['dow_hour_priors']
    dp_dm = dp[(dp['행정구'] == dist) & (dp['교통수단'] == mk)].drop(columns=['행정구', '교통수단'])
    fb_val = float(priors_bundle['cold_fallback'][(priors_bundle['cold_fallback']['행정구'] == dist) & (priors_bundle['cold_fallback']['교통수단'] == mk)]['point_prior_base_share'].iloc[0])

    dense = dense.merge(sp_dm, left_on='bucket_id', right_on='지점ID', how='left')
    dense['is_coldstart'] = dense['point_prior_base_share'].isna().astype(np.int8)
    dense['point_prior_base_share'] = dense['point_prior_base_share'].fillna(fb_val).astype(np.float32)
    dense['point_prior_rush_ratio'] = dense['point_prior_rush_ratio'].fillna(1.0).astype(np.float32)
    dense['point_prior_rain_elasticity'] = dense['point_prior_rain_elasticity'].fillna(1.0).astype(np.float32)
    dense['point_p95'] = dense['point_p95'].fillna(10.0).astype(np.float32)

    dense['is_wk_int'] = dense['is_weekend'].astype(np.int8)
    dense = dense.merge(
        dp_dm, left_on=['bucket_id', 'is_wk_int', '시간'], right_on=['지점ID', 'is_weekend', '시간'],
        how='left', suffixes=('', '_drop')
    )
    dense['point_prior_dow_hour_share'] = dense['point_prior_dow_hour_share'].fillna(dense['point_prior_base_share']).astype(np.float32)
    dense.drop(columns=[c for c in dense.columns if c.endswith('_drop') or c in ('지점ID', 'is_wk_int')], inplace=True)
    return dense


# 학습/검증 Parquet 생성 또는 캐시 로드
s2_paths = {en: (WORK_CACHE_DIR / f's2_train_{en}.parquet', WORK_CACHE_DIR / f's2_val_{en}.parquet') for en in MODES_KO.values()}
all_cached = all(tr.is_file() and va.is_file() for tr, va in s2_paths.values())

if all_cached and not FORCE_REBUILD_CACHE:
    print('📦 Drive 캐시에서 Stage 2 train/valid Parquet 확인 완료 (즉시 학습 가능)')
else:
    print('🛠️ 23~24년 원천 파일 스트리밍으로 Stage 2 밀집 데이터셋 구축 중...')
    t_b = time.time()
    rng = np.random.default_rng(SEED)
    train_files_all = sorted(glob.glob(str(DATA_DIR / '23_통합데이터' / POINT_SUBDIR / '2023_*.csv'))) + \
                      sorted(glob.glob(str(DATA_DIR / '24_통합데이터' / POINT_SUBDIR / '2024_*.csv')))

    # 파일당(총 50파일 중 구당 2개 연도) 샘플링 상한으로 균등 확보
    per_file_tr_cap = {en: max(MAX_TRAIN_ROWS_PER_MODE // 50, 10_000) for en in MODES_KO.values()}
    per_file_va_cap = {en: max(MAX_VALID_ROWS_PER_MODE // 25, 4_000) for en in MODES_KO.values()}

    buckets_tr = {en: [] for en in MODES_KO.values()}
    buckets_va = {en: [] for en in MODES_KO.values()}

    for i_f, fpath in enumerate(train_files_all, 1):
        fname = Path(fpath).name
        yr_str, dist_ext = fname.split('_', 1)
        dist = dist_ext.replace('.csv', '')
        raw_d = pd.read_csv(
            fpath, usecols=['기준_날짜', '시간', '교통수단', '지점ID', '이용량', '시간대_점유율(%)'],
            dtype={'지점ID': str}, encoding='utf-8'
        )
        for mk, en in MODES_KO.items():
            kept = priors_bundle['kept_points'][(dist, mk)]
            # train 구간
            tr_end_for_yr = TRAIN_END if yr_str == '2024' else '2023-12-31'
            d_tr = build_dense_district_mode_df(raw_d, dist, mk, kept, f'{yr_str}-01-01', tr_end_for_yr)
            if not d_tr.empty:
                cap = per_file_tr_cap[en]
                if len(d_tr) > cap:
                    d_tr = d_tr.sample(n=cap, random_state=SEED + i_f)
                buckets_tr[en].append(d_tr[POINT_SHARE_FEATURE_COLS + ['target_share']])
            # valid 구간 (2024 10~12월)
            if yr_str == '2024':
                d_va = build_dense_district_mode_df(raw_d, dist, mk, kept, VALID_START, VALID_END)
                if not d_va.empty:
                    cap_v = per_file_va_cap[en]
                    if len(d_va) > cap_v:
                        d_va = d_va.sample(n=cap_v, random_state=SEED + 100 + i_f)
                    buckets_va[en].append(d_va[POINT_SHARE_FEATURE_COLS + ['target_share']])
        if i_f % 10 == 0 or i_f == 50:
            print(f'  [{i_f:2d}/50] {fname} 처리 완료 ({time.time()-t_b:.0f}s)')

    for en, (p_tr, p_va) in s2_paths.items():
        df_tr_m = pd.concat(buckets_tr[en], ignore_index=True)
        df_va_m = pd.concat(buckets_va[en], ignore_index=True)
        df_tr_m.to_parquet(p_tr, index=False)
        df_va_m.to_parquet(p_va, index=False)
        print(f'  💾 [{en:6s}] train={len(df_tr_m):,}행 | valid={len(df_va_m):,}행 저장 완료')
    gc.collect()

---
### [Step 6] Stage 2 — 지점 점유율 LightGBM 모델 학습 + Optuna 튜닝
- 16차원 순수 수치형 피처(`POINT_SHARE_FEATURE_COLS`)로 수단별(`bike`, `bus`, `subway`) 독립 `LGBMRegressor` 를 학습합니다.

In [ ]:
# [Cell 6] Stage 2 지점 점유율 LightGBM 모델 학습 및 Optuna 튜닝
point_models = {}
point_best_params = {}

for en, (p_tr, p_va) in s2_paths.items():
    print(f'\n================ [Stage 2 · {en.upper()}] 학습 시작 ================')
    df_tr = pd.read_parquet(p_tr)
    df_va = pd.read_parquet(p_va)
    X_tr = df_tr[POINT_SHARE_FEATURE_COLS].astype(np.float32)
    y_tr = df_tr['target_share'].to_numpy(dtype=np.float64)
    X_va = df_va[POINT_SHARE_FEATURE_COLS].astype(np.float32)
    y_va = df_va['target_share'].to_numpy(dtype=np.float64)
    del df_tr, df_va

    best_p = {
        'learning_rate': 0.05, 'num_leaves': 63, 'max_depth': 8,
        'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.5, 'reg_lambda': 1.0,
    }
    if RUN_OPTUNA:
        def obj_s2(trial):
            cand = {
                'objective': 'regression', 'metric': 'mae', 'n_estimators': 400,
                'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.12, log=True),
                'num_leaves': trial.suggest_int('num_leaves', 31, 127),
                'max_depth': trial.suggest_int('max_depth', 5, 10),
                'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
                'subsample': trial.suggest_float('subsample', 0.65, 0.95),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.65, 1.0),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-2, 10.0, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-2, 10.0, log=True),
                'random_state': SEED, 'n_jobs': -1, 'verbose': -1,
            }
            mm = lgb.LGBMRegressor(**cand)
            mm.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(30, verbose=False)])
            return float(mean_absolute_error(y_va, np.clip(mm.predict(X_va), 0.0, None)))

        study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(obj_s2, n_trials=N_TRIALS_STAGE2)
        best_p = dict(study.best_params)
        print(f'  [{en.upper()} Best Trial] valid 점유율 MAE = {study.best_value:.5f}%p')

    final_m = lgb.LGBMRegressor(**best_p, objective='regression', metric='mae',
                                n_estimators=FINAL_N_ESTIMATORS, random_state=SEED, n_jobs=-1, verbose=-1)
    final_m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    va_pred = np.clip(final_m.predict(X_va), 0.0, None)
    print(f'  [{en.upper()} 완료] valid 점유율 MAE={mean_absolute_error(y_va, va_pred):.5f}%p | best_iter={getattr(final_m, "best_iteration_", FINAL_N_ESTIMATORS)}')
    point_models[en] = final_m
    point_best_params[en] = best_p
    del X_tr, y_tr, X_va, y_va
    gc.collect()

---
### [Step 7] 2025년 Test 25개 자치구 전수 평가 (4종 모델 × 6대 정량 지표) & 게이트 G2
2025년 25개 자치구 CSV를 1개씩 순차 스트리밍해 **샘플링 없는 100% 전수 Test 평가**를 수행합니다.
- **B0 (자치구 균등/일괄)**: 기존 자치구 묶음 계산의 한계 대변 (구 내 모든 지점에 균등 분배)
- **B1 (학습구간 기본 점유율)**: 지점별 학습구간 전체 평균 점유율
- **B2 (학습구간 요일×시간 조건부 프로필)**: 지점×주말×시간대 통계 프로필
- **M2 (LightGBM 지점 점유율 모델)**: 기상·캘린더·학습구간 프로필을 결합한 신규 AI 모델

모든 모델의 예측 점유율은 동일 `(날짜, 시간, 구, 수단)` 내에서 합이 `100%`가 되도록 비례 정규화한 뒤 Stage 1 자치구 예측 이용량을 곱해 **개별 지점 예측 이용량**을 복원합니다.

In [ ]:
# [Cell 7] 2025년 25개 자치구 100% 전수 평가 (4종 × 6대 정량 지표)
test_files_25 = sorted(glob.glob(str(DATA_DIR / '25_통합데이터' / POINT_SUBDIR / '2025_*.csv')))
assert len(test_files_25) == 25

MODEL_NAMES = ['B0_uniform', 'B1_base_prior', 'B2_dow_hour_prior', 'M2_lgb_point']

def init_acc():
    return {
        mn: {
            'n': 0, 'sum_y': 0.0, 'sum_abs_err': 0.0, 'sum_sq_err': 0.0,
            'sum_share_abs_err': 0.0, 'grade_ok': 0,
            'sum_y2': 0.0, 'spearman_list': [],
        }
        for mn in MODEL_NAMES
    }

acc = {en: init_acc() for en in MODES_KO.values()}
t_e = time.time()

def crowd_grade(vol: np.ndarray, p95: np.ndarray) -> np.ndarray:
    pct = np.clip((vol / np.maximum(p95, 1.0)) * 100.0, 0.0, 100.0)
    return np.where(pct < 45.0, 0, np.where(pct <= 75.0, 1, 2))

for idx_t, fpath in enumerate(test_files_25, 1):
    dist = Path(fpath).stem.split('_', 1)[1]
    raw_25 = pd.read_csv(
        fpath, usecols=['기준_날짜', '시간', '교통수단', '지점ID', '이용량', '시간대_점유율(%)'],
        dtype={'지점ID': str}, encoding='utf-8'
    )
    s1_d = stage1_test_df[stage1_test_df['구'] == dist].drop(columns=['구']).rename(columns={'날짜_str': '기준_날짜'})

    for mk, en in MODES_KO.items():
        kept = priors_bundle['kept_points'][(dist, mk)]
        dense = build_dense_district_mode_df(raw_25, dist, mk, kept, '2025-01-01', '2025-12-31', is_eval=True)
        if dense.empty:
            continue
        dense = dense.merge(s1_d[['기준_날짜', '시간', f'pred_{en}']], on=['기준_날짜', '시간'], how='left')
        dist_pred = dense[f'pred_{en}'].fillna(0.0).to_numpy(dtype=np.float64)
        y_true = dense['actual_usage'].to_numpy(dtype=np.float64)
        s_true = dense['target_share'].to_numpy(dtype=np.float64)
        p95_arr = dense['point_p95'].to_numpy(dtype=np.float64)
        g_true = crowd_grade(y_true, p95_arr)

        # 4종 원시 점유율 산출
        raw_shares = {
            'B0_uniform': np.ones(len(dense), dtype=np.float64),
            'B1_base_prior': np.clip(dense['point_prior_base_share'].to_numpy(dtype=np.float64), 0.0, None),
            'B2_dow_hour_prior': np.clip(dense['point_prior_dow_hour_share'].to_numpy(dtype=np.float64), 0.0, None),
            'M2_lgb_point': np.clip(point_models[en].predict(dense[POINT_SHARE_FEATURE_COLS].astype(np.float32)), 0.0, None),
        }

        # (기준_날짜, 시간) 그룹별 비례 정규화 → 합 = 100.0%
        grp_codes, _ = pd.factorize(dense['기준_날짜'].astype(str) + '_' + dense['시간'].astype(str))
        n_grp = int(grp_codes.max()) + 1
        cnt_grp = np.bincount(grp_codes, minlength=n_grp)[grp_codes]
        # Spearman 순위 상관용 대표 시간/일자 마스크 (매월 1·10·20일 × 08/14/18시)
        rho_mask = dense['시간'].isin([8, 14, 18]).to_numpy() & dense['기준_날짜'].astype(str).str.slice(8, 10).isin(['01', '10', '20']).to_numpy()
        rho_grp = grp_codes[rho_mask]
        rho_yt = y_true[rho_mask]

        # 각 모델 지표 누적 (WAPE/MAE/RMSE/등급/R²는 2025년 100% 전체 전수 계산)
        for mn, rs in raw_shares.items():
            grp_sum = np.bincount(grp_codes, weights=rs, minlength=n_grp)[grp_codes]
            if np.any(grp_sum <= 1e-12):
                s_norm = np.where(grp_sum > 1e-12, rs / np.maximum(grp_sum, 1e-12) * 100.0, 100.0 / cnt_grp)
            else:
                s_norm = rs / grp_sum * 100.0

            y_pred = dist_pred * (s_norm / 100.0)
            err = y_true - y_pred

            a = acc[en][mn]
            a['n'] += len(y_true)
            a['sum_y'] += float(y_true.sum())
            a['sum_y2'] += float((y_true ** 2).sum())
            a['sum_abs_err'] += float(np.abs(err).sum())
            a['sum_sq_err'] += float((err ** 2).sum())
            a['sum_share_abs_err'] += float(np.abs(s_true - s_norm).sum())
            a['grade_ok'] += int((crowd_grade(y_pred, p95_arr) == g_true).sum())

            rho_yp = y_pred[rho_mask]
            sub_df = pd.DataFrame({'grp': rho_grp, 'yt': rho_yt, 'yp': rho_yp})
            for _, g in sub_df.groupby('grp', sort=False):
                if g['yt'].std() > 0 and g['yp'].std() > 0:
                    rho = spearmanr(g['yt'], g['yp'])[0]
                    if not np.isnan(rho):
                        a['spearman_list'].append(float(rho))
        del dense
    del raw_25
    gc.collect()

    if idx_t % 5 == 0 or idx_t == 25:
        print(f'  [{idx_t:2d}/25] {dist} 평가 완료 ({time.time()-t_e:.0f}s)')

# 최종 집계 및 표 출력
eval_summary = {'generated_at': datetime.now().isoformat(timespec='seconds'), 'stage1': stage1_report, 'stage2': {}}
print('\n' + '=' * 95)
print('📊 2025년 Test 25개 자치구 전수 평가 결과 (6대 정량 지표)')
print('=' * 95)

for en in MODES_KO.values():
    eval_summary['stage2'][en] = {}
    print(f'\n▶ [{en.upper()}] (n={acc[en]["M2_lgb_point"]["n"]:,} 셀)')
    print(f'{"모델":20s} | {"WAPE":>8s} | {"MAE(명)":>9s} | {"점유율MAE(%p)":>12s} | {"RMSE(명)":>10s} | {"Spearman ρ":>11s} | {"등급일치율":>9s} | {"R²":>7s}')
    print('-' * 95)
    for mn in MODEL_NAMES:
        a = acc[en][mn]
        n = max(a['n'], 1)
        mean_y = a['sum_y'] / n
        sst = max(a['sum_y2'] - n * (mean_y ** 2), 1e-9)
        res = {
            'wape_pct': a['sum_abs_err'] / max(a['sum_y'], 1.0) * 100.0,
            'mae_volume': a['sum_abs_err'] / n,
            'mae_share_pp': a['sum_share_abs_err'] / n,
            'rmse_volume': float(np.sqrt(a['sum_sq_err'] / n)),
            'spearman_median': float(np.median(a['spearman_list'])) if a['spearman_list'] else 0.0,
            'grade_acc_pct': a['grade_ok'] / n * 100.0,
            'r2': 1.0 - (a['sum_sq_err'] / sst),
        }
        eval_summary['stage2'][en][mn] = res
        print(f"{mn:20s} | {res['wape_pct']:7.2f}% | {res['mae_volume']:9.2f} | {res['mae_share_pp']:12.4f} | "
              f"{res['rmse_volume']:10.2f} | {res['spearman_median']:11.4f} | {res['grade_acc_pct']:8.2f}% | {res['r2']:7.4f}")

# 게이트 G2 판정
g2_results = {}
for en in MODES_KO.values():
    m2 = eval_summary['stage2'][en]['M2_lgb_point']
    b2 = eval_summary['stage2'][en]['B2_dow_hour_prior']
    b0 = eval_summary['stage2'][en]['B0_uniform']
    rel_imp_b2 = (b2['wape_pct'] - m2['wape_pct']) / max(b2['wape_pct'], 1e-6) * 100.0
    rel_imp_b0 = (b0['wape_pct'] - m2['wape_pct']) / max(b0['wape_pct'], 1e-6) * 100.0
    passed = (rel_imp_b2 >= 5.0 and m2['spearman_median'] >= 0.80)
    g2_results[en] = {'rel_wape_gain_vs_B2_pct': rel_imp_b2, 'rel_wape_gain_vs_B0_pct': rel_imp_b0, 'g2_pass': passed}
    print(f"\n[게이트 G2 · {en.upper()}] M2 vs B0(자치구균등) WAPE 개선 {rel_imp_b0:+.2f}% | vs B2(조건부프로필) {rel_imp_b2:+.2f}% | Spearman ρ={m2['spearman_median']:.4f} → {'✅ PASS(M2)' if passed else '📌 FALLBACK(B2 권장)'}")
eval_summary['g2'] = g2_results

---
### [Step 8] ONNX 변환 (`16차원` 지점 모델 & `32차원` 자치구 모델) + 리포트 & 스키마 반출
- `onnxmltools` Opset 14로 변환 후 `onnxruntime` 추론 수치 정합성(`abs_diff < 1e-3`)을 검증합니다.
- 산출물은 Google Drive(`OUT_DIR`)에 백업하고, 필요 시 로컬 다운로드합니다.

In [ ]:
# [Cell 8] ONNX 변환, 추론 정합 검증, 스키마/리포트 저장 및 다운로드
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# 1. Stage 1 (32차원) ONNX 변환
s1_onnx_paths = []
for en, model in stage1_models.items():
    p = OUT_DIR / f'lgb_{en}_demand_v3.onnx'
    om = onnxmltools.convert_lightgbm(model, initial_types=[('float_input', FloatTensorType([None, 32]))], target_opset=14)
    p.write_bytes(om.SerializeToString())
    sess = ort.InferenceSession(str(p), providers=['CPUExecutionProvider'])
    sample_x = X_s1_te.iloc[:100].to_numpy(dtype=np.float32)
    diff = np.max(np.abs(sess.run(None, {sess.get_inputs()[0].name: sample_x})[0].ravel() - model.predict(sample_x)))
    print(f'[Stage1 ONNX] {p.name:28s} | dim={sess.get_inputs()[0].shape[1]} | size={p.stat().st_size/1024:7.1f}KB | max_diff={diff:.2e}')
    s1_onnx_paths.append(p)

# 2. Stage 2 (16차원) ONNX 변환
s2_onnx_paths = []
for en, model in point_models.items():
    p = OUT_DIR / f'lgb_{en}_point_share.onnx'
    om = onnxmltools.convert_lightgbm(model, initial_types=[('float_input', FloatTensorType([None, 16]))], target_opset=14)
    p.write_bytes(om.SerializeToString())
    sess = ort.InferenceSession(str(p), providers=['CPUExecutionProvider'])
    sample_x = pd.read_parquet(s2_paths[en][1], columns=POINT_SHARE_FEATURE_COLS).iloc[:100].to_numpy(dtype=np.float32)
    diff = np.max(np.abs(sess.run(None, {sess.get_inputs()[0].name: sample_x})[0].ravel() - model.predict(sample_x)))
    print(f'[Stage2 ONNX] {p.name:28s} | dim={sess.get_inputs()[0].shape[1]} | size={p.stat().st_size/1024:7.1f}KB | max_diff={diff:.2e}')
    s2_onnx_paths.append(p)

# 3. 지점 사전 프로필 Parquet + 스키마/평가 리포트 JSON & MD 저장
priors_bundle['static_priors'].to_parquet(OUT_DIR / 'point_static_priors.parquet', index=False)
priors_bundle['dow_hour_priors'].to_parquet(OUT_DIR / 'point_dow_hour_priors.parquet', index=False)

schema_v3 = {
    'model_version': 'v3.0-point-share-colab',
    'stage1_feature_dim': 32,
    'stage1_feature_cols': FEATURE_COLS_32,
    'stage2_feature_dim': 16,
    'stage2_feature_cols': POINT_SHARE_FEATURE_COLS,
    'forbidden_leakage_cols_verified': sorted(FORBIDDEN_LEAKAGE_COLS),
    'split': {'train': f'2023-01-01~{TRAIN_END}', 'valid': f'{VALID_START}~{VALID_END}', 'test': '2025-01-01~2025-12-31'},
    'stage2_best_params': point_best_params,
    'evaluation': eval_summary,
}
(OUT_DIR / 'point_share_schema.json').write_text(json.dumps(schema_v3, ensure_ascii=False, indent=2), encoding='utf-8')
eval_json = RESULTS_DIR / f'point_share_eval_{stamp}.json'
eval_json.write_text(json.dumps(eval_summary, ensure_ascii=False, indent=2), encoding='utf-8')

md_lines = [
    f'# 📊 지점 단위(정류소·대여소·역) 수요 분배 모델 평가 리포트 (`{stamp}`)',
    '',
    '> **목적**: 자치구별 일괄 계산 한계 보완 → 개별 정류소·대여소·역 단위 독립 수요·혼잡도 예측',
    '',
    '## Stage 2 (2025 전수 Test · 6대 정량 지표)',
    '',
    '| 수단 | 모델 | WAPE | MAE(명) | 점유율 MAE(%p) | RMSE(명) | Spearman ρ | 등급 일치율 | R² |',
    '| :-: | :--- | -: | -: | -: | -: | -: | -: | -: |',
]
for en in MODES_KO.values():
    for mn in MODEL_NAMES:
        r = eval_summary['stage2'][en][mn]
        md_lines.append(
            f"| {en} | {mn} | {r['wape_pct']:.2f}% | {r['mae_volume']:.2f} | {r['mae_share_pp']:.4f} | "
            f"{r['rmse_volume']:.2f} | {r['spearman_median']:.4f} | {r['grade_acc_pct']:.2f}% | {r['r2']:.4f} |"
        )
eval_md = RESULTS_DIR / f'point_share_eval_{stamp}.md'
eval_md.write_text('\n'.join(md_lines) + '\n', encoding='utf-8')

print(f'\n✅ 모든 산출물 저장 완료 → {OUT_DIR}')
print(f'✅ 평가 리포트 저장 완료 → {eval_md}')

if IN_COLAB and files is not None:
    for dl_path in [eval_json, eval_md, OUT_DIR / 'point_share_schema.json'] + s2_onnx_paths:
        files.download(str(dl_path))